In [ ]:
import pandas as pd
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import distances
import matplotlib.pyplot as plt
import itertools
from itertools import *
import seaborn as sns
import collections
from statistics import mean

import warnings
warnings.filterwarnings('ignore')

In [ ]:
(5*1000000/0.004) - 498087000


In [ ]:
752500000.0 / (5*1000000/0.004) + .398

In [ ]:
u = mda.Universe('output.gro','trajectory.dcd')

In [ ]:
ca = u.select_atoms('name CA and resid 2')
n_ca = len(ca)
n_ca

In [ ]:
self_distances = distances.self_distance_array(ca.positions)
self_distances

In [ ]:
aa_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
combos = list(itertools.combinations_with_replacement(aa_list, 2))
len(combos)

In [ ]:
ARG = u.select_atoms('name CA and resid 2 and resname ARG')
ALA = u.select_atoms('name CA and resid 2 and resname ALA')
ASN = u.select_atoms('name CA and resid 2 and resname ASN')
ASP = u.select_atoms('name CA and resid 2 and resname ASP')
CYS = u.select_atoms('name CA and resid 2 and resname CYS')
GLU = u.select_atoms('name CA and resid 2 and resname GLU')
GLN = u.select_atoms('name CA and resid 2 and resname GLN')
GLY = u.select_atoms('name CA and resid 2 and resname GLY')
HIS = u.select_atoms('name CA and resid 2 and resname HIS')
ILE = u.select_atoms('name CA and resid 2 and resname ILE')
LEU = u.select_atoms('name CA and resid 2 and resname LEU')
LYS = u.select_atoms('name CA and resid 2 and resname LYS')
MET = u.select_atoms('name CA and resid 2 and resname MET')
PHE = u.select_atoms('name CA and resid 2 and resname PHE')
PRO = u.select_atoms('name CA and resid 2 and resname PRO')
SER = u.select_atoms('name CA and resid 2 and resname SER')
THR = u.select_atoms('name CA and resid 2 and resname THR')
TRP = u.select_atoms('name CA and resid 2 and resname TRP')
TYR = u.select_atoms('name CA and resid 2 and resname TYR')
VAL = u.select_atoms('name CA and resid 2 and resname VAL')

In [ ]:
import MDAnalysis as mda
import MDAnalysis.lib.distances as ld

def get_atom_self_distances(pdb, xtc, selection='all', first_frame=0, last_frame=-1, step=1):
    """
    Load distances between all selected atoms.
    Parameters
    ----------
    pdb : str
        File name for the reference file (PDB or GRO format).
    xtc : str
        File name for the trajectory (xtc format).
    selection : str
        Selection string to choose which atoms to include. Default: all.
    first_frame : int, default=0
        First frame to return of the features. Zero-based.
    last_frame : int, default=-1
        Last frame to return of the features. Zero-based.
    step : int, default=1
        Subsampling step width when reading the frames.
    Returns
    -------
    feature_names : list of str
        Names of all distances
    features_data : numpy array
        Data for all distances [Å]
    """

    u = mda.Universe(pdb, xtc)
    a = u.select_atoms(selection)
    num_at = len(a)

    # Name the atoms
    at_labels = [atom.residue.resname for atom in a]

    # Name the distance labels
    d_labels = []
    k = -1
    for i in range(num_at):
        for j in range(i + 1, num_at):
            k += 1
            _dl = '%s-%s'%(at_labels[i], at_labels[j])
            d_labels.append(_dl)

    # Calculate the distances
    num_at = len(a)
    num_dist = int(num_at*(num_at-1)/2)
    len_traj = len(u.trajectory)
    template = np.zeros([num_dist,])
    data_arr = np.zeros([len_traj,num_dist])
    frame = 0
    
    for ts in u.trajectory[first_frame:last_frame:step]:
        data_arr[frame] = ld.self_distance_array(a.positions, result=template)
        frame += 1

    return d_labels, data_arr


def get_calpha_distances2(pdb, xtc, first_frame=0, last_frame=-1, step=1):
    """ Load distances between all C-alpha atoms"""
    names, data =  get_atom_self_distances(pdb, xtc, 
                                           selection='name CA and resid 2', 
                                           first_frame=first_frame, 
                                           last_frame=last_frame,                                                 
                                           step=step)
    return names, data

In [ ]:
names, data = get_calpha_distances2('output.gro','trajectory.dcd')

In [ ]:
len(combos)

In [ ]:
combos[0][0]+'-'+combos[0][1]

In [ ]:
def get_avg_dists(u, frame, combo):
    
    sel_1 = u.select_atoms('name CA and resid 2 and resname ' + combo[0])
    sel_2 = u.select_atoms('name CA and resid 2 and resname ' + combo[1])
    
    for ts in u.trajectory[frame]:
    
        dist_list = list((distances.distance_array(sel_1.positions, sel_2.positions, box=u.dimensions)).flatten())
        dist_list = [i for i in dist_list if i != 0]
        dist_sum = sum(dist_list)
        avg = dist_sum / len(dist_list)
        
        return avg

In [ ]:
get_avg_dists(u, 0, combos[4])

In [ ]:
import pandas as pd
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import distances
import matplotlib.pyplot as plt
import collections
from statistics import mean
import itertools
from itertools import *
import seaborn as sns
import csv
import warnings
warnings.filterwarnings('ignore')

def get_avg_dists(u, frame, combo):
    
    sel_1 = u.select_atoms('name CA and resid 2 and resname ' + combo[0])
    sel_2 = u.select_atoms('name CA and resid 2 and resname ' + combo[1])
    
    for ts in u.trajectory[frame]:
    
        dist_list = list((distances.distance_array(sel_1.positions, sel_2.positions, box=u.dimensions)).flatten())
        dist_list = [i for i in dist_list if i != 0]
        dist_sum = sum(dist_list)
        avg = dist_sum / len(dist_list)
        
        return avg


u = mda.Universe('output.gro','trajectory.dcd')

aa_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

# all possible residue pair combinations
combos = list(itertools.combinations_with_replacement(aa_list, 2))
    
# hold the average distances for all the frames
master_dict = {(combo[0],combo[1]): [] for combo in combos}

for frame in range(len(u.trajectory[0:10])):
    for combo in combos:
        name = (combo[0],combo[1])

        master_dict[name].append(get_avg_dists(u, frame, combo))
        
    if (frame%5==0):
        with open('master_dict.csv', 'w') as csv_file:  
            writer = csv.writer(csv_file)
            for key, value in master_dict.items():
                writer.writerow([key, value])
        
avg_dist_dict = {k:mean(v) for k,v in master_dict.items()}
        
ser = pd.DataFrame(list(avg_dist_dict.values()), 
                index=pd.MultiIndex.from_tuples(avg_dist_dict.keys()))
df = ser.unstack().fillna(0)

ctmap = df.values + df.values.T

ctmap =ctmap / np.mean(ctmap)

di = np.diag_indices(20)

ctmap[di] = ctmap[di]/2 

fig, ax  = plt.subplots()
sns.heatmap(ctmap,ax=ax)
ax.set_xticks(list(np.arange(20)));
ax.invert_yaxis()

In [ ]:
ser = pd.DataFrame(list(avg_dist_dict.values()), 
                index=pd.MultiIndex.from_tuples(avg_dist_dict.keys()))
df = ser.unstack().fillna(0)

ctmap = df.values + df.values.T

ctmap =ctmap / np.mean(ctmap)

di = np.diag_indices(20)

ctmap[di] = ctmap[di]/2 

fig, ax  = plt.subplots()
sns.heatmap(ctmap,ax=ax)
ax.set_xticks(list(np.arange(20)));
ax.invert_yaxis()

In [ ]:
csv = pd.read_csv('avg-dist-data.csv')
df = pd.DataFrame(csv)
final_df = df.drop(axis=1, columns=['Unnamed: 0'])
final_df_2 = final_df / df.values.max()
hm = sns.heatmap(final_df, yticklabels=aa_list, xticklabels=aa_list, cmap="YlGnBu")
hm.invert_yaxis()